# 13 - Quantum Fourier Transform

Implement the QFT — the foundation of Shor's algorithm and phase estimation.

**Concepts:** QFT, controlled phase rotations, bit reversal

In [ ]:
import quantsdk as qs
import math

In [ ]:
def qft(n: int) -> qs.Circuit:
    """Build an n-qubit QFT circuit."""
    circuit = qs.Circuit(n, name=f"QFT-{n}")
    
    for i in range(n):
        circuit.h(i)
        for j in range(i + 1, n):
            angle = math.pi / (2 ** (j - i))
            circuit.cp(i, j, angle)
    
    # Bit reversal
    for i in range(n // 2):
        circuit.swap(i, n - 1 - i)
    
    return circuit

def inverse_qft(n: int) -> qs.Circuit:
    """Build an n-qubit inverse QFT circuit."""
    circuit = qs.Circuit(n, name=f"iQFT-{n}")
    
    for i in range(n // 2):
        circuit.swap(i, n - 1 - i)
    
    for i in range(n - 1, -1, -1):
        for j in range(n - 1, i, -1):
            angle = -math.pi / (2 ** (j - i))
            circuit.cp(i, j, angle)
        circuit.h(i)
    
    return circuit

In [ ]:
# QFT on |000> (should stay |000> since FT of delta at 0 is uniform)
circuit = qs.Circuit(3)
# Apply QFT
qft_circ = qft(3)
for gate in qft_circ.gates:
    circuit._gates.append(gate)  # Append gates
circuit.measure_all()

result = qs.run(circuit, shots=2000, seed=42)
print("QFT|000> (should be uniform):")
for bs, count in sorted(result.counts.items()):
    print(f"  |{bs}>: {count/2000:.3f}")

In [ ]:
# QFT then iQFT should be identity
n = 3
circuit = qs.Circuit(n)
circuit.x(0).x(2)  # Prepare |101>
# Apply QFT gates
for gate in qft(n).gates:
    circuit._gates.append(gate)
# Apply iQFT gates
for gate in inverse_qft(n).gates:
    circuit._gates.append(gate)
circuit.measure_all()

result = qs.run(circuit, shots=100, seed=42)
print(f"QFT then iQFT on |101>: {result.most_likely} (should be 101)")

In [ ]:
# Circuit stats
for n in [2, 3, 4, 5]:
    circuit = qft(n)
    print(f"QFT-{n}: {circuit.gate_count} gates, depth {circuit.depth}")